Friends By Age, Using DataFrame
Print a dataframe of the name and age for each entry
of fakefriends-header.csv


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkSQL").getOrCreate()

friends = spark.read.csv("../fakefriends.csv", header=True, inferSchema=True)

print('Here is our inferred scheme')

friends.printSchema()




Getting all fake friends over 18, get count by age of friends, Exercise with Ethan

In [ ]:
from pyspark.sql import Row, SparkSession

spark = SparkSession.builder \
    .appName("GetFriendTotals") \
    .getOrCreate()  

# Returning a Row object. And then manually cast the fields to the correct type
# A better way to do this is to infer a schema
def parseLine(line):
    fields = line.split(',')
    return Row(ID=int(fields[0]), name=str(fields[1].encode('utf-8')), age=int(fields[2]), num_friends=int(fields[3]))

# Make an RDD
lines = spark.sparkContext.textFile('../fakefriends.csv')
# Use map() function
people = lines.map(parseLine)
#Create dataframe
schemaPeople = spark.createDataFrame(people).cache() # improves performance if need to do multiple things with this object
# The cache() makes it so that the program doesn't need to rebuild the DataFrame 
schemaPeople.createOrReplaceTempView('people')

adults = spark.sql("SELECT * FROM people WHERE age >= 18") # Makes use of view made from schemaPeople

for p in adults.collect():
    print(p)

schemaPeople.groupBy('age').count().orderBy('age').show() # MAkes use of schemaPeople again

spark.stop()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/01 14:39:54 WARN Utils: Your hostname, CartersDell, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/01 14:39:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 14:39:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Row(ID=0, name="b'Will'", age=33, num_friends=385)
Row(ID=1, name="b'Jean-Luc'", age=33, num_friends=2)
Row(ID=2, name="b'Hugh'", age=55, num_friends=221)
Row(ID=3, name="b'Deanna'", age=40, num_friends=465)
Row(ID=4, name="b'Quark'", age=68, num_friends=21)
+---+-----+
|age|count|
+---+-----+
| 17|    1|
| 33|    2|
| 40|    1|
| 55|    1|
| 68|    1|
+---+-----+



Another exercise with fake friends

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('AnotherAgeExample').getOrCreate()

people = spark.read.option('header', True).option('inferschedema', True).csv('../fakefriends-header.csv')

print('Here is our inferred schema:')
people.printSchema()

people.select('name').show()
people.filter(people.age <= 35).show()
people.groupBy('age').count().show()

people.select(people.name, people.age + 10).show()
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/01 14:33:32 WARN Utils: Your hostname, CartersDell, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/01 14:33:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 14:33:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Here is our inferred schema:
root
 |-- userID: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- friends: string (nullable = true)

+--------+
|    name|
+--------+
|    Will|
|Jean-Luc|
|    Hugh|
|  Deanna|
|   Quark|
|  Weyoun|
|  Gowron|
|    Will|
|  Jadzia|
|    Hugh|
|     Odo|
|     Ben|
|   Keiko|
|Jean-Luc|
|    Hugh|
|     Rom|
|  Weyoun|
|     Odo|
|Jean-Luc|
|  Geordi|
+--------+
only showing top 20 rows
+------+--------+---+-------+
|userID|    name|age|friends|
+------+--------+---+-------+
|     0|    Will| 33|    385|
|     1|Jean-Luc| 26|      2|
|     9|    Hugh| 27|    181|
|    16|  Weyoun| 22|    323|
|    17|     Odo| 35|     13|
|    21|   Miles| 19|    268|
|    22|   Quark| 30|     72|
|    24|  Julian| 25|      1|
|    25|     Ben| 21|    445|
|    26|  Julian| 22|    100|
|    32|     Nog| 26|    281|
|    35| Beverly| 27|    305|
|    36|  Kasidy| 32|     81|
|    39|    Morn| 31|    192|
|    44|   Nerys| 

Challenge, friends by age with dataframes, print DF of the name and age for each entry of fakefriends-header.csv

ON THE SLIDES 66

In [ ]:
from pyspark.sql import SparkSession, functions as funk

spark = SparkSession.builder.appName('FriendsAge&Name').getOrCreate()

people = spark.read.option('header', True).option('inferschedema', True).csv('../fakefriends-header.csv')

print('Here is our inferred schema:')
people.printSchema()

people.select('name', 'age').show()
people.groupBy('age').avg('friends').sort('age').show()
people
people.select(people.name, people.age + 10).show()
spark.stop()

2nd exercise with ethan, fake friends
In notes with the @udf decorator

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName('UDFExample').getOrCreate()

@udf(returnType = StringType())
def user_details(name, age):
    return f"Name: {name}, Age: {age}"

